# Feature Engineering

Analysons les colonnes, les dupliquées, variables NaN, etc.

In [2]:
import pandas as pd
import numpy as np

In [17]:
df = pd.read_csv("data/Phishing_Email.csv")

# Classification
df = df.replace("Safe Email", 0)
df = df.replace("Phishing Email", 1)

# On enlève la colonne des IDs
df = df.drop(df.columns[0], axis=1)

# On renomme les colonnes
df = df.rename(columns={"Email Text":"content", "Email Type": "class"})

# On enlève les variables NaN
df = df.dropna()

# Suppression dupliquées
df = df.drop_duplicates()

# On regarde la répartition du dataset
df["class"].value_counts()

C:\Users\vtnde\AppData\Local\Temp\ipykernel_51404\3206614742.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("Phishing Email", 1)


class
0    10980
1     6558
Name: count, dtype: int64

On remarque un léger déséquilibre au niveau du volume de chaque class, un algorithme d'oversampling pourrait être intéressant. Mais attaquons-nous d'abord à l'aspect NLP, appliquons TF-IDF pour évaluer l'importance d'un mot dans le dataset.

Note: On oublie pas de faire le train_test_split avant pour éviter les data leakages

In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df["content"], df['class'], test_size=0.2, random_state=42)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Max 5000 colonnes, le mot doit apparaître dans 5 lignes minimum
tfidf = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.8)
result = tfidf.fit_transform(X_train)
tfidf.get_feature_names_out()

X_test = tfidf.transform(X_test)

array(['00', '000', '0000', ..., 'zone', 'zur', '½ï'],
      shape=(5000,), dtype=object)

On se retrouve avec 5000 colonnes (initialement 163224 colonnes sans les limites), c'est énorme. Pour éviter le surapprentissage, il faut réduire le nombre de colonne (réduire la dimension), on peut opter pour un PCA mais cela nous fait perdre en clarté au niveau du fonctionnement du modèle, si on cherche à expliquer pourquoi ça marche plus tard, on ne pourra à cause de PCA. On va donc opter pour des régulatisations L1 ou L2.

In [46]:
from xgboost import XGBClassifier
model_xgb = XGBClassifier(
    n_estimators=100,
    reg_alpha=10,
    reg_lambda=1,
    max_depth=6,
    scale_pos_weight=y_train.value_counts()[0]/y_train.value_counts()[1]
)

# Exemple CatBoost
from catboost import CatBoostClassifier
model_catboost = CatBoostClassifier(
    iterations=100,
    l2_leaf_reg=5,
    depth=6
)

In [37]:
# SMOTE

from imblearn.over_sampling import SMOTE

ratio = (y_train.value_counts()[1] * 1.05)/y_train.value_counts()[0]
print(ratio)
sm = SMOTE(sampling_strategy=ratio)
X_train_res, y_train_res = sm.fit_resample(result, y_train)

print(f"Avant SMOTE : {y_train.value_counts()}")
print(f"Après SMOTE : {y_train_res.value_counts()}")

0.6303353484658378
Avant SMOTE : class
0    8767
1    5263
Name: count, dtype: int64
Après SMOTE : class
0    8767
1    5526
Name: count, dtype: int64


Le dataset est suffisamment équilibré, faire de l'undersampling n'est pas nécessaire, même l'oversampling n'était pas nécessaire dans notre situation.

In [48]:
# Prédiction!

# XGB sans SMOTE
model_xgb.fit(result, y_train)
y_pred_xgb = model_xgb.predict(X_test)

# XGB avec SMOTE
model_xgb.fit(X_train_res, y_train_res)
y_pred_xgb_SMOTE = model_xgb.predict(X_test)

# Catboost sans SMOTE
model_catboost.fit(result, y_train)
y_pred_cat = model_catboost.predict(X_test.toarray())

0:	learn: 0.6725368	total: 267ms	remaining: 26.4s
1:	learn: 0.6492868	total: 497ms	remaining: 24.3s
2:	learn: 0.6264729	total: 752ms	remaining: 24.3s
3:	learn: 0.6076001	total: 983ms	remaining: 23.6s
4:	learn: 0.5946944	total: 1.21s	remaining: 23s
5:	learn: 0.5782108	total: 1.43s	remaining: 22.4s
6:	learn: 0.5653386	total: 1.66s	remaining: 22.1s
7:	learn: 0.5531166	total: 1.96s	remaining: 22.5s
8:	learn: 0.5398069	total: 2.19s	remaining: 22.1s
9:	learn: 0.5278976	total: 2.41s	remaining: 21.7s
10:	learn: 0.5187949	total: 2.64s	remaining: 21.4s
11:	learn: 0.5115459	total: 2.89s	remaining: 21.2s
12:	learn: 0.5029215	total: 3.12s	remaining: 20.9s
13:	learn: 0.4908272	total: 3.35s	remaining: 20.6s
14:	learn: 0.4821920	total: 3.59s	remaining: 20.4s
15:	learn: 0.4755044	total: 3.84s	remaining: 20.2s
16:	learn: 0.4690736	total: 4.07s	remaining: 19.9s
17:	learn: 0.4617110	total: 4.31s	remaining: 19.6s
18:	learn: 0.4557722	total: 4.54s	remaining: 19.4s
19:	learn: 0.4497899	total: 4.79s	remaining

In [49]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

def afficher_stats(y_true, y_pred, nom_modele):
    print(f"--- Stats pour {nom_modele} ---")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print("\nRapport détaillé :")
    print(classification_report(y_true, y_pred))
    print("\nMatrice de confusion :")
    print(confusion_matrix(y_true, y_pred))
    print("-" * 30)

# Affichage
afficher_stats(y_test, y_pred_xgb, "XGBoost")
afficher_stats(y_test, y_pred_xgb_SMOTE, "XGBoost (avec SMOTE)")
afficher_stats(y_test, y_pred_cat, "CatBoost")

--- Stats pour XGBoost ---
Accuracy: 0.9632

Rapport détaillé :
              precision    recall  f1-score   support

           0       0.98      0.96      0.97      2213
           1       0.93      0.97      0.95      1295

    accuracy                           0.96      3508
   macro avg       0.96      0.97      0.96      3508
weighted avg       0.96      0.96      0.96      3508


Matrice de confusion :
[[2119   94]
 [  35 1260]]
------------------------------
--- Stats pour XGBoost (avec SMOTE) ---
Accuracy: 0.9644

Rapport détaillé :
              precision    recall  f1-score   support

           0       0.98      0.96      0.97      2213
           1       0.93      0.97      0.95      1295

    accuracy                           0.96      3508
   macro avg       0.96      0.97      0.96      3508
weighted avg       0.97      0.96      0.96      3508


Matrice de confusion :
[[2122   91]
 [  34 1261]]
------------------------------
--- Stats pour CatBoost ---
Accuracy: 0.9

XGBoost gagne, SMOTE n'a eu aucun impact important sur le modèle final car le dataset était suffisamment équilibré.

Pour aller plus loin, on peut faire du tuning des hyperparamètres, mais on considère que la précision est satisfaisante actuellement.

Maintenant on enregistre les modèles (TF-IDF + XGBoost) pour les utiliser autre part

In [52]:
import os
import joblib

joblib.dump(tfidf, "models/tfidf.joblib")
joblib.dump(model_xgb, "models/xgb_model.joblib")

['models/xgb_model.joblib']